# Анализ на чувствителността


Металообработващо предприятие произвежда два вида отливки: стоманени и чугунени. Едно от ограниченията в производствения процес е времето за работа на две от машините, които обработват отливките. Една стоманена отливка изисква 2 часа на първата машина и 1 час на втората машина, докато една чугунена отливка изисква 1 час на първата машина и три часа на втората машина. Всяка от машините може да работи най-много по 8 часа на ден. Печалбата от продажбата на една стоманена отливка е 130 EUR, а от една чугунена отливка - 100 EUR. Целта е да се максимизира общата печалба от производството на отливките.

[Desmos](https://www.desmos.com/calculator/)

1. Какъв производствен план ще препоръчате на предприятието, за да максимизира печалбата си?
2. Кои ограничения са активни в оптимума?
3. Управителят на предприятието иска от вас препоръка дали да наеме още една машина от вид 1 за 8 часа на ден срещу 50 евро на час.
4. Предприятието обмисля дали да въведе нов продукт, за чието производство (за една единица) са нужни по един час от всяка машина. Каква трябва да е печалбата на единица от новия продукт, за да си струва въвеждането му?
5. В какви интервали могат да варират коефициентите във функцията на печалба без това да промени оптималния производствен план? При анализа варирайте само един от коефициентите, а другите оставете на първоначалните им стойности.

Добавете следното ограничение: за всеки час работа на машина 1 са нужни 15 минути за поддръжка след края на работния ден. За машина 2 са нужни 20 мин. Техническият отдел може да отдели до 4 часа за тази дейност.

1. Изчислете оптималния производствен план, като съобразите и новото ограничение.
2. Кои са активните ограничения?
3. Струва ли си наемането на един допълнителен час за поддръжка на цена от 40 евро на ден?


In [7]:
import gurobipy as gp
from gurobipy import GRB

# Create model
model = gp.Model("metal_castings")

# Decision variables: x1 = steel castings, x2 = cast iron castings
x1 = model.addVar(name="steel", lb=0)
x2 = model.addVar(name="cast_iron", lb=0)

# Objective: maximize profit
model.setObjective(130 * x1 + 100 * x2, GRB.MAXIMIZE)

# Constraints
# Machine 1: 2*x1 + 1*x2 <= 8 hours
model.addConstr(2 * x1 + x2 <= 8, "machine_1")

# Machine 2: 1*x1 + 3*x2 <= 8 hours
model.addConstr(x1 + 3 * x2 <= 8, "machine_2")

# model.addConstr((5/6) * x1 + (5/4) * x2 <= 4, "Maintenance")
# Solve
model.optimize()

# Print results
if model.status == GRB.OPTIMAL:
    print(f"Optimal solution found:")
    print(f"Steel castings (x1): {x1.X:.2f}")
    print(f"Cast iron castings (x2): {x2.X:.2f}")
    print(f"Maximum profit: {model.ObjVal:.2f} EUR")

    # Print shadow prices (dual values) for the constraints
    print("\nShadow prices (dual values):")
    for constr in model.getConstrs():
        print(f"{constr.ConstrName}: {constr.Pi:.2f}")

    # Print allowable increases and decreases for the right-hand side of the constraints
    print("\nAllowable increases and decreases for the right-hand side of the constraints:")
    for constr in model.getConstrs():
        print(f"{constr.ConstrName}: Lower = {constr.SARHSLow}, Upper = {constr.SARHSUp}")


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i9-14900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros (Max)
Model fingerprint: 0xc7c8ddaf
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+00, 8e+00]

Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.3000000e+32   2.500000e+30   2.300000e+02      0s
       2    5.7600000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.760000000e+02
Optimal solution found:
Steel castings (x1): 3.20
Cast iron castings (x2): 1.60
Maximum profit: 576.00 EUR

Shadow prices

In [8]:
model.addConstr((5/6) * x1 + (5/4) * x2 <= 4, "Maintenance")
model.optimize()

# Print results
if model.status == GRB.OPTIMAL:
    print(f"Optimal solution found:")
    print(f"Steel castings (x1): {x1.X:.2f}")
    print(f"Cast iron castings (x2): {x2.X:.2f}")
    print(f"Maximum profit: {model.ObjVal:.2f} EUR")

    # Print shadow prices (dual values) for the constraints
    print("\nShadow prices (dual values):")
    for constr in model.getConstrs():
        print(f"{constr.ConstrName}: {constr.Pi:.2f}")

    # Print allowable increases and decreases for the right-hand side of the constraints
    print("\nAllowable increases and decreases for the right-hand side of the constraints:")
    for constr in model.getConstrs():
        print(f"{constr.ConstrName}: Lower = {constr.SARHSLow}, Upper = {constr.SARHSUp}")


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i9-14900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros (Max)
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [8e-01, 3e+00]
  Objective range  [1e+02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+00, 8e+00]

LP warm-start: use basis

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.7600000e+02   6.666667e-01   0.000000e+00      0s
       1    5.4800000e+02   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.480000000e+02
Optimal solution found:
Steel castings (x1): 3.60
Cast iron castings (x2): 0.80
Maximum profit: 548.00 EUR

Shadow prices (dual values):
machine_1: 47.50
machine_2: 0.00
Maintenance: 42.00